# EDA - Adult (Census Income) Dataset

**Goal:** Predict whether annual income > $50K
**Task type:** Binary classification (**Class 1 = >50K**, **Class 0 = ≤50K**).  
**What this notebook covers (Step 1: EDA):** understanding the data, visualizing distributions/relationships, and explaining findings.

In [1]:
# 1) Imports
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load the data
Make sure you've extracted the dataset into the working directory as `adult.csv`.

In [ ]:
# 2) Load dataset (adjust the path if needed)
df = pd.read_csv('adult.csv')
df.head()



## 2. Basic structure and data types
Understanding the size, column types, and a quick peek at the top rows.


In [ ]:

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())



## 3. Missing values and special placeholders
The Adult dataset often uses `'?'` to denote missing values in some categorical columns (e.g., `workclass`, `occupation`, `native-country`). We count both real `NaN` and `'?'` placeholders.


In [ ]:

# Standard missing values
print("Missing values per column (NaN):")
print(df.isna().sum())

# Count '?' placeholders by column
print("\nCount of '?' per column:")
question_mark_counts = {col: int((df[col] == ' ?').sum()) for col in df.columns if df[col].dtype == 'object'}
print(pd.Series(question_mark_counts))



## 4. Unique values and categorical cardinality
Helps identify which columns are categorical and how many categories they have.


In [ ]:

unique_counts = df.nunique().sort_values(ascending=False)
unique_counts



## 5. Descriptive statistics (numerical features)
Gives a quick sense of central tendency and spread; useful to spot skewness and potential outliers.


In [ ]:

df.describe(numeric_only=True)



## 6. Target distribution
We check for class imbalance between `>50K` and `<=50K`.


In [ ]:

target_counts = df['income'].value_counts().sort_index()
fig = plt.figure(figsize=(5,4))
plt.bar(target_counts.index, target_counts.values)
plt.title('Income Distribution')
plt.xlabel('Income')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Short interpretation
print("Most records are '<=50K', indicating class imbalance (roughly ~75/25 in many versions of this dataset).")



## 7. Numerical feature distributions
Histograms show the shape (skew, multimodality) of each numeric column.


In [ ]:

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for col in numeric_cols:
    fig = plt.figure(figsize=(5,3))
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()



## 8. Outliers (boxplots)
Boxplots give a compact view of outliers and overall spread.


In [ ]:

for col in ['age','hours-per-week','capital-gain','capital-loss']:
    if col in df.columns:
        fig = plt.figure(figsize=(5,3))
        plt.boxplot(df[col].dropna(), vert=False)
        plt.title(f'Boxplot — {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()



## 9. Categorical feature distributions
We visualize the frequency of each category to understand dominant classes and sparsity.


In [ ]:

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'income']

for col in cat_cols:
    vc = df[col].value_counts()
    fig = plt.figure(figsize=(7,4))
    plt.barh(vc.index[::-1], vc.values[::-1])
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()



## 10. Correlation (numerical features)
A heatmap of Pearson correlations helps spot linear relationships between numeric variables.


In [ ]:

corr = df.select_dtypes(include=['int64','float64']).corr(numeric_only=True)
fig = plt.figure(figsize=(7,6))
plt.imshow(corr, interpolation='nearest')
plt.title('Correlation Heatmap (Numerical)')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.tight_layout()
plt.show()

corr



## 11. Relationships with target — numeric features
Comparing numeric distributions between the two income classes.


In [ ]:

# Map target to binary 0/1 for convenience
y_map = {' <=50K': 0, '<=50K': 0, ' >50K': 1, '>50K': 1}
y = df['income'].map(y_map)

for col in [c for c in numeric_cols if c != 'fnlwgt']:
    # Prepare data arrays by class
    class0 = df.loc[y==0, col].dropna()
    class1 = df.loc[y==1, col].dropna()
    if len(class0) > 0 and len(class1) > 0:
        fig = plt.figure(figsize=(6,4))
        plt.boxplot([class0, class1], labels=['<=50K','>50K'])
        plt.title(f'{col} vs Income')
        plt.xlabel('Income Class')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()



## 12. Relationships with target — categorical features
We compute the share of `>50K` in each category (per column) and plot it. This reveals which groups have higher positive rates.


In [ ]:

def positive_rate_by_category(frame, feature, target_col='income', positive_value='>50K'):
    # Support both ' >50K' and '>50K'
    pos_variants = {positive_value, f' {positive_value}'}
    rates = []
    for cat, sub in frame.groupby(feature):
        total = len(sub)
        pos = sub[target_col].isin(pos_variants).sum()
        rate = pos / total if total else 0.0
        rates.append((cat, rate, total))
    res = pd.DataFrame(rates, columns=[feature, 'positive_rate', 'count']).sort_values('positive_rate', ascending=False)
    return res

for col in cat_cols:
    pr = positive_rate_by_category(df, col)
    # Only plot if reasonable number of categories
    if 2 <= len(pr) <= 40:
        fig = plt.figure(figsize=(8, max(3, 0.3*len(pr))))
        plt.barh(pr[col][::-1], pr['positive_rate'][::-1])
        plt.title(f'P(>50K) by {col}')
        plt.xlabel('Share of >50K')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()
        display(pr.head(10))



## 13. Takeaways (EDA Summary)
- **Data shape & schema:** ~48K rows, 14 features + 1 target. Mix of integer and categorical fields.
- **Missingness:** some `NaN`, plus `'?'` placeholders in `workclass`, `occupation`, `native-country` → will be handled in preprocessing.
- **Target imbalance:** many more `<=50K` than `>50K` (roughly ~75/25). Keep in mind for model evaluation and sampling strategies.
- **Numerical features:** `capital-gain` and `capital-loss` are highly skewed (many zeros, few large values). `age` and `hours-per-week` show intuitive patterns.
- **Correlations:** modest linear correlations overall; `education-num` often relates to income and education categories.
- **Relationships:** Higher `age`, `hours-per-week`, certain occupations (e.g., managerial/professional), and higher education levels tend to have a higher share of `>50K`.
  
**Next (Step 2 — Preprocessing):** handle `'?'` as missing, impute/clean values, encode categoricals, scale/transform skewed numerics, and address class imbalance (e.g., stratified splits, class weights).
